In [1]:
import importlib

import torch

from classes.BirdDataset import *

from common.common_utils import *

In [2]:
location = "data/soundscape_recordings_amazon/"

data_folder = "recordings/"
annotations = "annotations_divided_overfit"

In [3]:
import json

with open(location + "label_encodings.json", "r") as f:
    encodings = json.load(f)

label_to_idx = encodings["label_to_idx"]
idx_to_label = {int(k): v for k, v in encodings["idx_to_label"].items()}

In [ ]:
precompute_sed_samples(
    annotations_file=location + annotations,
    recordings_dir=location + data_folder,
    output_dir=location + data_folder + annotations + ".precomputed_cap5/",
    sr=32000, hop_length=256, n_mels=128, window_len=5.0,
    fmin=0.0, fmax=None,
    label_to_idx=label_to_idx,
    num_workers=15
)

precompute_sed_samples(
    annotations_file=location + annotations,
    recordings_dir=location + "recordings_clean/",
    output_dir=location + "recordings_clean/" + annotations + ".precomputed_cap5/",
    sr=32000, hop_length=256, n_mels=128, window_len=5.0,
    fmin=0.0, fmax=None,
    label_to_idx=label_to_idx,
    num_workers=15
)

Random-window SED:   0%|          | 0/21 [00:00<?, ?it/s]

In [4]:
train_dataset = BirdDataset(
    data_path = location + data_folder,
    annotations_file = location + annotations,
    split = "val",
    label_to_idx = label_to_idx,
    target_type = "tf",
    precomputed = True,
    precomputed_dir = location + data_folder + annotations + ".precomputed/",
)

val_dataset = BirdDataset(
    data_path = location + data_folder,
    annotations_file = location + annotations,
    split = "val",
    label_to_idx = label_to_idx,
    target_type = "tf",
    include_overlaps = True,
    precomputed = False,
)


print(len(train_dataset), len(val_dataset))

280 280


In [5]:
import pandas as pd

annotations_df = pd.read_csv(location + annotations + ".csv", sep=",")

In [5]:
show_sample_idx = 56

In [7]:
# 1) Visualize an existing dataset window
rows = plot_window_with_annotations(ds=train_dataset, annotations_df=annotations_df, idx=show_sample_idx)

# 2) Find a random window with >= n samples (fully contained) and visualize it
idx, fname, ws, we, rows = find_random_window_with_n_samples(
    ds=train_dataset,
    annotations_df=annotations_df,
    n=2,
    mode="at_least",
    fully_contained=True,
    seed=498
)
if idx is not None:
    plot_window_with_annotations(ds=train_dataset, annotations_df=annotations_df, idx=idx)

print(fname, ws, we)

,Filename,Start Time (s),End Time (s),Low Freq (Hz),High Freq (Hz),Species eBird Code
336,PER_002_S02_20190116_100007Z.flac,1798.6,1799.7,754,2539,butwoo1


c:\Users\Markoobah\Desktop\Deep Learning Subject\Project\orniwatch_dp_project\classes\BirdDataset.py:167: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(os.

FileNotFoundError: [Errno 2] No such file or directory: 'data/soundscape_recordings_amazon/recordings/annotations_divided_overfit.precomputed/000336.pt'

In [4]:
from torch.utils.data import ConcatDataset

train_dataset = BirdDataset(
    data_path = location + data_folder,
    annotations_file = location + annotations,
    label_to_idx = label_to_idx,
    target_type = "tf",
    precomputed = True,
    window_len=5.0,
    precomputed_dir = location + data_folder + annotations + ".precomputed_cap5/",
)
train_clean_dataset = BirdDataset(
    data_path = location + "recordings_clean/", 
    annotations_file = location + annotations,
    label_to_idx = label_to_idx,
    target_type = "tf",
    precomputed = True,
    window_len=5.0,
    precomputed_dir=location + "recordings_clean/" + annotations + ".precomputed_cap5/",
    )
train_dataset = ConcatDataset([train_dataset, train_clean_dataset])

val_dataset = BirdDataset(
    data_path = location + data_folder,
    annotations_file = location + annotations,
    label_to_idx = label_to_idx,
    split = "val",
    target_type = "tf",
    precomputed = True,
    window_len=5.0,
    precomputed_dir=location + data_folder + annotations + ".precomputed_cap5",
    )

print(len(train_dataset), len(val_dataset))

578 271


In [5]:
batch_size = 8

In [6]:
trainloader = torch.utils.data.DataLoader(
    train_dataset, 
    batch_size=batch_size,
    shuffle=True, 
    num_workers=os.cpu_count() // 2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=6,
)

valloader = torch.utils.data.DataLoader(
    val_dataset, 
    batch_size=batch_size,
    shuffle=False, 
    num_workers=os.cpu_count() // 2,
    pin_memory=True,
    persistent_workers=True,
    prefetch_factor=6,
)

In [7]:
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0))

True
NVIDIA GeForce RTX 4070


In [8]:
from classes.BirdModels import BirdCRNN
from torchinfo import summary

model = BirdCRNN(n_mels=128, n_classes=len(label_to_idx), cnn_channels=256, hidden_size=1024, gru_num_layers=4)

mel, target = train_dataset[0]
B, C, F, T = 1, *mel.shape

summary(
    model,
    input_size=(B, C, F, T)
)

c:\Users\Markoobah\Desktop\Deep Learning Subject\Project\orniwatch_dp_project\classes\BirdDataset.py:153: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  blob = torch.load(sel

Layer (type:depth-idx)                   Output Shape              Param #
BirdCRNN                                 [1, 28, 625]              --
├─Sequential: 1-1                        [1, 256, 32, 625]         --
│    └─Conv2d: 2-1                       [1, 64, 128, 625]         960
│    └─BatchNorm2d: 2-2                  [1, 64, 128, 625]         128
│    └─ReLU: 2-3                         [1, 64, 128, 625]         --
│    └─MaxPool2d: 2-4                    [1, 64, 64, 625]          --
│    └─Dropout: 2-5                      [1, 64, 64, 625]          --
│    └─Conv2d: 2-6                       [1, 128, 64, 625]         122,880
│    └─BatchNorm2d: 2-7                  [1, 128, 64, 625]         256
│    └─ReLU: 2-8                         [1, 128, 64, 625]         --
│    └─MaxPool2d: 2-9                    [1, 128, 32, 625]         --
│    └─Dropout: 2-10                     [1, 128, 32, 625]         --
│    └─Conv2d: 2-11                      [1, 128, 32, 625]         147,456
│ 

In [9]:
import torch.optim as optim
import torch.nn as nn

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=0.001)

In [10]:
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

BirdCRNN(
  (cnn): Sequential(
    (0): Conv2d(1, 64, kernel_size=(5, 3), stride=(1, 1), padding=(2, 1), bias=False)
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (4): Dropout(p=0.2, inplace=False)
    (5): Conv2d(64, 128, kernel_size=(5, 3), stride=(1, 1), padding=(2, 1), bias=False)
    (6): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (7): ReLU(inplace=True)
    (8): MaxPool2d(kernel_size=(2, 1), stride=(2, 1), padding=0, dilation=1, ceil_mode=False)
    (9): Dropout(p=0.2, inplace=False)
    (10): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (11): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (12): ReLU(inplace=True)
    (13): Dropout(p=0.2, inplace=False)
    (14): Conv2d(128, 256, kernel_size=(3, 3

In [32]:
# stats = benchmark_pipeline(model, trainloader, device=device, batches=50, use_amp=True)

In [15]:
pos_weight, raw_ratio = compute_pos_weight(
    trainloader, device, limit_batches=None, power=0.5, cap=(0.5, 8.0), normalize=True
)
print("raw neg/pos median:", float(raw_ratio.median().cpu()))
print("tempered pos_weight:", pos_weight.tolist())

criterion_tf   = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)
criterion_time = torch.nn.BCEWithLogitsLoss(pos_weight=pos_weight)

KeyboardInterrupt: 

In [11]:
import gc
os.environ["CUDA_LAUNCH_BLOCKING"] = "1"
torch.cuda.empty_cache()
gc.collect()

97

In [20]:
# Sanity overfit
import time, math
from torch import nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_TF   = True
USE_AMP  = True
AMP_DTYPE = torch.float16

lam, steps, tol = 0.3, 200, 1e-3

xb, yb = next(iter(trainloader))
xb, yb = xb.to(device).float(), yb.to(device).float()
want_tf = (yb.dim()==4) and USE_TF

model.to(device).train()
#model.apply(lambda m: setattr(m, "p", 0.0) if isinstance(m, nn.Dropout) else None)

opt = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=0.0)
bce = nn.BCEWithLogitsLoss()
scaler = torch.amp.GradScaler('cuda', enabled=USE_AMP)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

best = math.inf
for s in range(1, steps+1):
    t0 = time.perf_counter()
    opt.zero_grad(set_to_none=True)
    with torch.amp.autocast('cuda', enabled=USE_AMP, dtype=AMP_DTYPE):
        time_logits, tf_logits = model(xb, return_tf=want_tf)
        loss = (bce(tf_logits, yb) + lam*bce(time_logits, yb.amax(dim=2))) if want_tf else bce(time_logits, yb)
    scaler.scale(loss).backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0)
    scaler.step(opt); scaler.update()
    torch.cuda.synchronize() if torch.cuda.is_available() else None
    dt = (time.perf_counter() - t0) * 1000

    best = min(best, loss.item())
    if s % 10 == 0 or loss.item() <= tol:
        if want_tf:
            with torch.no_grad():
                tfm  = ((tf_logits.sigmoid()>0.5)==(yb>0.5)).float().mean().item()
                tim  = ((time_logits.sigmoid()>0.5)==(yb.amax(dim=2)>0.5)).float().mean().item()
        alloc = torch.cuda.memory_allocated()/1e6 if torch.cuda.is_available() else 0
        reserv = torch.cuda.memory_reserved()/1e6  if torch.cuda.is_available() else 0
        if want_tf:
            print(f"[{s}] {dt:.1f} ms | loss={loss.item():.4f} | tf={tfm*100:.1f}% | time={tim*100:.1f}% | alloc={alloc:.0f}MB resv={reserv:.0f}MB")
        else:
            with torch.no_grad():
                tim  = ((time_logits.sigmoid()>0.5)==(yb>0.5)).float().mean().item()
            print(f"[{s}] {dt:.1f} ms | loss={loss.item():.4f} | time={tim*100:.1f}% | alloc={alloc:.0f}MB resv={reserv:.0f}MB")
        if loss.item() <= tol:
            print("✅ memorized"); break

[10] 411.3 ms | loss=0.1386 | tf=99.9% | time=99.3% | alloc=2489MB resv=4933MB
[20] 412.0 ms | loss=0.0179 | tf=99.9% | time=99.3% | alloc=2489MB resv=4933MB
[30] 410.7 ms | loss=0.0142 | tf=99.9% | time=99.3% | alloc=2489MB resv=4933MB
[40] 412.8 ms | loss=0.0132 | tf=99.9% | time=99.3% | alloc=2489MB resv=4933MB
[50] 414.3 ms | loss=0.0124 | tf=99.9% | time=99.3% | alloc=2489MB resv=4933MB
[60] 411.2 ms | loss=0.0118 | tf=99.9% | time=99.3% | alloc=2489MB resv=4933MB
[70] 413.9 ms | loss=0.0110 | tf=99.9% | time=99.3% | alloc=2489MB resv=4933MB
[80] 411.8 ms | loss=0.0086 | tf=99.9% | time=99.5% | alloc=2489MB resv=4933MB
[90] 410.9 ms | loss=0.0082 | tf=99.9% | time=99.5% | alloc=2489MB resv=4933MB
[100] 410.8 ms | loss=0.0076 | tf=99.9% | time=99.5% | alloc=2489MB resv=4933MB
[110] 415.5 ms | loss=0.0071 | tf=99.9% | time=99.5% | alloc=2489MB resv=4933MB
[120] 415.1 ms | loss=0.0052 | tf=99.9% | time=99.5% | alloc=2489MB resv=4933MB
[130] 409.7 ms | loss=0.0045 | tf=99.9% | time=99

In [12]:
import scripts.train_script as TS

from common.logging_extensions import now_ts

In [13]:
cfg = TS.TrainConfig(
    location=location,
    data_folder=data_folder,
    annotations_file=annotations,
    precomputed=True,
    precomputed_dirs = [
        "recordings/" + annotations + ".precomputed_cap5/",
        "recordings_clean/" + annotations + ".precomputed_cap5/",
    ],
    n_mels=128,
    target_type="tf",
    batch_size=8,
    epochs=10,
    lr=1e-3,
    num_workers=os.cpu_count() // 2,
    amp="off",
    pos_weight="auto",
    model=model,
    seed=4894,
    logs_base="logs/",
    run_name="checkTrain_" + now_ts(),
)

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

In [14]:
importlib.reload(TS)

TS.fit(cfg, label_to_idx)

[DEBUG] train samples: 577, val samples: 275


2025-11-08 16:48:55 | INFO | pos_weight(auto): tensor([0.9736, 0.9846, 0.7214, 0.9249, 1.3044, 0.6991, 1.2049, 0.6495, 0.9023,
        1.3050, 0.6535, 0.7083, 0.8428, 1.3285, 0.7650, 1.3082, 1.2057, 0.6609,
        1.0048, 0.7800, 1.0562, 0.7951, 0.8817, 3.0533, 0.7300, 0.8429, 0.8943,
        0.8191], device='cuda:0')
2025-11-08 16:48:55 | INFO | config: {
  "location": "data/soundscape_recordings_amazon/",
  "data_folder": "recordings/",
  "annotations_file": "annotations_divided_overfit",
  "precomputed": true,
  "precomputed_dirs": [
    "recordings/annotations_divided_overfit.precomputed_cap5/",
    "recordings_clean/annotations_divided_overfit.precomputed_cap5/"
  ],
  "n_mels": 128,
  "window_len": 10.0,
  "batch_size": 8,
  "num_workers": 8,
  "target_type": "tf",
  "include_overlaps": true,
  "lr": 0.001,
  "epochs": 10,
  "model_name": "BirdCRNN",
  "model": "BirdCRNN",
  "amp": "off",
  "pos_weight": "auto",
  "save_top_k": 3,
  "monitor": "val_f1",
  "monitor_mode": "max",



[DEBUG] tf_logits mean/std: -0.006490837782621384 0.08262229710817337
[DEBUG] y_tf mean/std: 0.0018549665110185742 0.04302935674786568
[DEBUG] time_logits mean/std: -0.0037153528537601233 0.06331752985715866
[DEBUG] y_time mean/std: 0.010292856954038143 0.1009306013584137


2025-11-08 16:49:51 | INFO | metrics
2025-11-08 16:49:51 | INFO | epoch 001 | {'epoch': 1, 'train_loss': 0.073155, 'val_loss': 0.035862, 'val_f1': 0.185994, 'val_f1_macro': 0.185994, 'val_prec': 0.0844, 'val_rec': 0.861279, 'time_f1': 0.833994, 'tf_f1': 0.185994} | 55.4s
2025-11-08 16:49:55 | INFO | checkpoint improved on 'val_f1' -> saved (score=0.185994)



[DEBUG] tf_logits mean/std: -13.631965637207031 13.390270233154297
[DEBUG] y_tf mean/std: 0.0008374999742954969 0.02892746962606907
[DEBUG] time_logits mean/std: -4.643008708953857 1.0879652500152588
[DEBUG] y_time mean/std: 0.005035714246332645 0.07078411430120468


2025-11-08 16:50:45 | INFO | metrics
2025-11-08 16:50:45 | INFO | epoch 002 | {'epoch': 2, 'train_loss': 0.033278, 'val_loss': 0.033937, 'val_f1': 0.669589, 'val_f1_macro': 0.669589, 'val_prec': 0.739796, 'val_rec': 0.861225, 'time_f1': 0.861225, 'tf_f1': 0.669589} | 50.2s
2025-11-08 16:50:49 | INFO | checkpoint improved on 'val_f1' -> saved (score=0.669589)



[DEBUG] tf_logits mean/std: -11.768203735351562 11.722431182861328
[DEBUG] y_tf mean/std: 0.002277790103107691 0.047671813517808914
[DEBUG] time_logits mean/std: -4.836617469787598 0.8918191194534302
[DEBUG] y_time mean/std: 0.01209999993443489 0.1093328669667244


2025-11-08 16:51:39 | INFO | metrics
2025-11-08 16:51:39 | INFO | epoch 003 | {'epoch': 3, 'train_loss': 0.032505, 'val_loss': 0.034854, 'val_f1': 0.308372, 'val_f1_macro': 0.308372, 'val_prec': 0.245918, 'val_rec': 0.861225, 'time_f1': 0.861225, 'tf_f1': 0.308372} | 50.3s
2025-11-08 16:51:43 | INFO | checkpoint improved on 'val_f1' -> saved (score=0.308372)



[DEBUG] tf_logits mean/std: -10.977428436279297 10.270389556884766
[DEBUG] y_tf mean/std: 0.0018544642953202128 0.043023545295000076
[DEBUG] time_logits mean/std: -4.399603843688965 0.8708917498588562
[DEBUG] y_time mean/std: 0.010392856784164906 0.10141458362340927


KeyboardInterrupt: 

In [29]:
model.__class__.__name__

'BirdCRNN'